# Anchor-based detectors — shared official-validation benchmark

**Faster R-CNN (two-stage) vs RetinaNet (one-stage)**  
This notebook restores the original trained checkpoints **without retraining**, evaluates them on the same official `val` split using the original **ignore-aware, per-class COCO evaluator**, and measures inference latency on **the same Colab GPU**.

**Comparison candidates:** Faster R-CNN baseline/tuned; RetinaNet baseline/smaller anchors. Other focal-loss experiments are already documented in the RetinaNet training notebook. Fold-0 checkpoints are used consistently; do not select checkpoints based on the official validation results.

**Important:** Official validation is used here for reporting, not for further hyperparameter selection. The benchmark may take a long time because it evaluates all 2,846 images with a dense ignore-aware COCO protocol. All prediction chunks are cached on Drive and resumed after interruption.

In [13]:
%pip -q install pycocotools pandas tqdm pillow matplotlib
%pip install -q kaggle

from google.colab import drive
drive.mount('/content/drive')
import os, gc, io, json, math, time, hashlib, contextlib, tempfile, copy, platform
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch, torchvision
from torch import nn
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms.functional import pil_to_tensor
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

assert torch.cuda.is_available(), 'Enable a GPU runtime in Colab first.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(DEVICE))
print('Torch:', torch.__version__, 'TorchVision:', torchvision.__version__)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Torch: 2.14.1+cu130 TorchVision: 0.29.1+cu130


In [14]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("kaggle_token").strip()

print("Kaggle token loaded from Colab Secrets.")

Kaggle token loaded from Colab Secrets.


In [15]:
!kaggle competitions download \
    -c multiple-object-video-detection-hw-2-ucu-ai-2026 \
    -p /content/detection_data

multiple-object-video-detection-hw-2-ucu-ai-2026.zip: Skipping, found more recently modified local copy (use --force to force download)


In [16]:
from pathlib import Path
import zipfile

DATA_ROOT = Path("/content/detection_data")

archive = DATA_ROOT / "multiple-object-video-detection-hw-2-ucu-ai-2026.zip"

assert archive.is_file(), f"ZIP archive not found: {archive}"

print("Extracting dataset...")

with zipfile.ZipFile(archive, "r") as zf:
    zf.extractall(DATA_ROOT)

print("Extraction completed.")

GT_PATH = DATA_ROOT / "val" / "annotations_coco.json"

assert GT_PATH.is_file(), (
    f"Validation annotations not found: {GT_PATH}"
)

print("Validation dataset is ready!")

Extracting dataset...
Extraction completed.
Validation dataset is ready!


## 1. Paths and evaluation protocol

Place the two Faster R-CNN checkpoints under `ucu_ai_hw2/faster_rcnn_runs/` as agreed. The extracted dataset is expected at `/content/detection_data` (as in the RetinaNet notebook) and must contain `val/annotations_coco.json` and `val/<relative image path>`. Change `DATA_ROOT` if your extracted dataset is elsewhere. We check all paths before running.

In [17]:
DRIVE_ROOT = Path('/content/drive/MyDrive/ucu_ai_hw2')
DATA_ROOT = Path('/content/detection_data')  # Extracted dataset location used in 03_retinanet notebook
OUTPUT_DIR = DRIVE_ROOT / 'benchmarks' / 'anchor_based_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS = {
    'Faster R-CNN baseline': DRIVE_ROOT / 'faster_rcnn_runs/faster_rcnn_f15f44c2db51_fold0/best.pt',
    'Faster R-CNN tuned': DRIVE_ROOT / 'faster_rcnn_runs/faster_rcnn_6bb40977bf33_fold0/best.pt',
    'RetinaNet baseline': DRIVE_ROOT / 'retinanet_runs/retinanet_a67e4f8ebd8a_fold0/best.pt',
    'RetinaNet smaller anchors': DRIVE_ROOT / 'retinanet_runs/retinanet_ce67efb8abd1_fold0/best.pt',
}
for name, path in CHECKPOINTS.items():
    print(f'{name}:', 'OK' if path.is_file() else f'MISSING: {path}')
assert all(p.is_file() for p in CHECKPOINTS.values()), 'Upload/check missing checkpoints.'
GT_PATH = DATA_ROOT / 'val/annotations_coco.json'
assert GT_PATH.is_file(), (
    f'Missing official validation annotations: {GT_PATH}\n'
    'The Colab runtime may have restarted. Run the dataset download/extraction '
    'cells from 03_retinanet first, or set DATA_ROOT to the actual extracted dataset directory.'
)
with GT_PATH.open(encoding='utf-8') as fh:
    official_val_coco = json.load(fh)
val_table = pd.DataFrame(official_val_coco['images']).rename(columns={'id':'image_id'}).sort_values('image_id').reset_index(drop=True)
assert val_table.image_id.is_unique and len(val_table)==2846
assert {x['id'] for x in official_val_coco['categories']} == set(range(1,9))
print('Official validation images:', len(val_table))
print('First image:', DATA_ROOT/'val'/val_table.iloc[0].file_name)
assert (DATA_ROOT/'val'/val_table.iloc[0].file_name).is_file(), 'Set DATA_ROOT to the extracted dataset root.'

MAX_DETS = 100  # matches the original COCO evaluator
BENCHMARK_IMAGES = 64
WARMUP = 5
CHUNK_FRAMES = 32


Faster R-CNN baseline: OK
Faster R-CNN tuned: OK
RetinaNet baseline: OK
RetinaNet smaller anchors: OK
Official validation images: 2846
First image: /content/detection_data/val/images/uav0000086_00000_v/0000001.jpg


## 2. Reconstruct the exact original model architectures

The following definitions are copied from the uploaded `02-faster-cnn(3).ipynb` and `03_retinanet(3).ipynb`, including ignore-aware training classes, configurable GIoU RoI heads, anchors, and focal-loss parameters. We initialize **without downloading pretrained weights** and then load the complete trained `model` state dict. The ignore-aware customizations do not alter the normal eval forward pass.

In [18]:
# Original Faster R-CNN definitions, isolated from RetinaNet definitions
faster_ns = {'torch': torch, 'nn': nn, 'math': math}
FASTER_SOURCE = 'from torchvision.models.detection.transform import GeneralizedRCNNTransform, resize_boxes\nfrom torchvision.models.detection.rpn import RegionProposalNetwork\nfrom torchvision.models.detection.roi_heads import RoIHeads, fastrcnn_loss\nfrom torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\nfrom torchvision.ops import generalized_box_iou_loss\nimport torch.nn.functional as F\nIGNORE_IOA_THRESHOLD=0.5\n\ndef max_ignore_ioa(candidate_boxes, ignore_boxes, chunk_size=4096):\n    """Maximum intersection / candidate area over ignored boxes."""\n    candidates = candidate_boxes.float()\n    ignored = ignore_boxes.to(\n        device=candidates.device,\n        dtype=torch.float32,\n    )\n\n    overlaps = candidates.new_zeros(len(candidates))\n\n    if len(candidates) == 0 or len(ignored) == 0:\n        return overlaps\n\n    for start in range(0, len(candidates), chunk_size):\n        boxes = candidates[start:start + chunk_size]\n\n        top_left = torch.maximum(\n            boxes[:, None, :2], ignored[None, :, :2]\n        )\n        bottom_right = torch.minimum(\n            boxes[:, None, 2:], ignored[None, :, 2:]\n        )\n        intersection = (\n            (bottom_right - top_left).clamp(min=0).prod(dim=-1)\n        )\n        area = (\n            (boxes[:, 2:] - boxes[:, :2])\n            .clamp(min=0)\n            .prod(dim=-1)\n        )\n\n        overlaps[start:start + len(boxes)] = (\n            intersection / area.clamp_min(1e-6)[:, None]\n        ).amax(dim=1)\n\n    return overlaps\n\ndef exclude_ignored_background(labels, boxes, ignored, threshold):\n    overlaps = max_ignore_ioa(boxes, ignored)\n    updated = labels.clone()\n    updated[labels.eq(0) & overlaps.ge(threshold)] = -1\n    return updated\n\nclass IgnoreAwareTransform(GeneralizedRCNNTransform):\n    def resize(self, image, target=None):\n        original_size = image.shape[-2:]\n\n        if target is not None:\n            target = target.copy()\n            ignored = target.get("ignore_boxes")\n        else:\n            ignored = None\n\n        image, target = super().resize(image, target)\n\n        if ignored is not None:\n            target["ignore_boxes"] = resize_boxes(\n                ignored, original_size, image.shape[-2:]\n            )\n\n        return image, target\n\nclass IgnoreAwareRPN(RegionProposalNetwork):\n    ignore_ioa_threshold = IGNORE_IOA_THRESHOLD\n\n    def assign_targets_to_anchors(self, anchors, targets):\n        labels, matched_boxes = super().assign_targets_to_anchors(\n            anchors, targets\n        )\n\n        labels = [\n            exclude_ignored_background(\n                image_labels,\n                image_anchors,\n                target["ignore_boxes"],\n                self.ignore_ioa_threshold,\n            )\n            for image_labels, image_anchors, target\n            in zip(labels, anchors, targets)\n        ]\n\n        return labels, matched_boxes\n\nclass IgnoreAwareRoIHeads(RoIHeads):\n    ignore_ioa_threshold = IGNORE_IOA_THRESHOLD\n\n    def select_training_samples(self, proposals, targets):\n        self._active_ignore_boxes = [\n            target["ignore_boxes"] for target in targets\n        ]\n        try:\n            return super().select_training_samples(proposals, targets)\n        finally:\n            del self._active_ignore_boxes\n\n    def assign_targets_to_proposals(\n        self, proposals, gt_boxes, gt_labels\n    ):\n        matched_indices, labels = (\n            super().assign_targets_to_proposals(\n                proposals, gt_boxes, gt_labels\n            )\n        )\n\n        ignored_per_image = getattr(\n            self, "_active_ignore_boxes", None\n        )\n\n        if ignored_per_image is not None:\n            labels = [\n                exclude_ignored_background(\n                    image_labels,\n                    image_proposals,\n                    ignored,\n                    self.ignore_ioa_threshold,\n                )\n                for image_labels, image_proposals, ignored\n                in zip(labels, proposals, ignored_per_image)\n            ]\n\n        return matched_indices, labels\n\ndef compute_roi_losses(\n    class_logits,\n    box_regression,\n    labels,\n    regression_targets,\n    proposals,\n    box_coder,\n    loss_name,\n):\n    if loss_name == "smooth_l1":\n        # Preserve TorchVision\'s original baseline loss.\n        return fastrcnn_loss(\n            class_logits,\n            box_regression,\n            labels,\n            regression_targets,\n        )\n\n    if loss_name != "giou":\n        raise ValueError(f"Unknown RoI regression loss: {loss_name}")\n\n    flat_labels = torch.cat(labels, dim=0)\n\n    classification_loss = F.cross_entropy(\n        class_logits,\n        flat_labels,\n    )\n\n    positive_indices = torch.where(flat_labels > 0)[0]\n\n    if positive_indices.numel() == 0:\n        # Keep a differentiable zero when no positive proposals exist.\n        return classification_loss, box_regression.sum() * 0.0\n\n    # Decode class-specific predictions: [N, num_classes, 4].\n    predicted_boxes = box_coder.decode(\n        box_regression.float(),\n        proposals,\n    )\n\n    # Each regression target contains one encoded target box.\n    target_boxes = box_coder.decode(\n        torch.cat(regression_targets, dim=0).float(),\n        proposals,\n    ).reshape(-1, 4)\n\n    positive_classes = flat_labels[positive_indices]\n\n    selected_predictions = predicted_boxes[\n        positive_indices,\n        positive_classes,\n    ]\n    selected_targets = target_boxes[positive_indices]\n\n    regression_loss = generalized_box_iou_loss(\n        selected_predictions,\n        selected_targets,\n        reduction="sum",\n    ) / flat_labels.numel()\n\n    return classification_loss, regression_loss\n\nclass ConfigurableRoIHeads(IgnoreAwareRoIHeads):\n    roi_box_loss = "smooth_l1"\n\n    def forward(self, features, proposals, image_shapes, targets=None):\n        # Preserve the original baseline and the standard inference path.\n        if not self.training or self.roi_box_loss == "smooth_l1":\n            return super().forward(\n                features, proposals, image_shapes, targets\n            )\n\n        if self.roi_box_loss != "giou":\n            raise ValueError(\n                f"Unknown RoI regression loss: {self.roi_box_loss}"\n            )\n\n        if targets is None:\n            raise ValueError("Training requires targets.")\n\n        if self.has_mask() or self.has_keypoint():\n            raise NotImplementedError(\n                "The custom GIoU path supports bounding boxes only."\n            )\n\n        self.check_targets(targets)\n\n        for target in targets:\n            if target["labels"].dtype != torch.int64:\n                raise TypeError("Target labels must use torch.int64.")\n            if not target["boxes"].is_floating_point():\n                raise TypeError("Target boxes must be floating point.")\n\n        # Inherited sampling preserves our ignored-region handling.\n        proposals, _, labels, regression_targets = (\n            self.select_training_samples(proposals, targets)\n        )\n\n        box_features = self.box_roi_pool(\n            features, proposals, image_shapes\n        )\n        box_features = self.box_head(box_features)\n        class_logits, box_regression = self.box_predictor(box_features)\n\n        classification_loss, regression_loss = compute_roi_losses(\n            class_logits,\n            box_regression,\n            labels,\n            regression_targets,\n            proposals,\n            self.box_coder,\n            loss_name=self.roi_box_loss,\n        )\n\n        return [], {\n            "loss_classifier": classification_loss,\n            "loss_box_reg": regression_loss,\n        }\n\ndef build_faster_rcnn(config, pretrained=True):\n    model = fasterrcnn_resnet50_fpn_v2(\n        weights=(\n            FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT\n            if pretrained else None\n        ),\n        weights_backbone=None,\n        min_size=config["min_size"],\n        max_size=config["max_size"],\n        box_detections_per_img=config["box_detections_per_img"],\n        box_score_thresh=config["box_score_thresh"],\n        rpn_post_nms_top_n_train=config["rpn_post_nms_top_n_train"],\n        rpn_post_nms_top_n_test=config["rpn_post_nms_top_n_test"],\n        box_batch_size_per_image=config["box_batch_size_per_image"],\n        box_positive_fraction=config["box_positive_fraction"],\n    )\n\n    # Without pretrained weights, the constructor can change backbone\n    # trainability defaults. Match our pretrained baseline explicitly.\n    for name, parameter in model.backbone.body.named_parameters():\n        parameter.requires_grad_(\n            name.startswith(("layer2.", "layer3.", "layer4."))\n        )\n\n    in_features = model.roi_heads.box_predictor.cls_score.in_features\n    model.roi_heads.box_predictor = FastRCNNPredictor(\n        in_features,\n        config["num_classes"],\n    )\n\n    model.transform.__class__ = IgnoreAwareTransform\n    model.rpn.__class__ = IgnoreAwareRPN\n    model.roi_heads.__class__ = ConfigurableRoIHeads\n\n    model.rpn.ignore_ioa_threshold = config["ignore_ioa_threshold"]\n    model.roi_heads.ignore_ioa_threshold = config["ignore_ioa_threshold"]\n    model.roi_heads.roi_box_loss = config["roi_box_loss"]\n\n    return model'
exec(FASTER_SOURCE, faster_ns)
print('Faster R-CNN builder ready')


Faster R-CNN builder ready


In [19]:
# Original RetinaNet definitions, isolated from Faster R-CNN definitions
retina_ns = {'torch': torch, 'nn': nn, 'math': math}
RETINA_SOURCE = 'from torchvision.models.detection import _utils as detection_utils\nfrom torchvision.models.detection.transform import GeneralizedRCNNTransform, resize_boxes\nfrom torchvision.ops import box_iou, sigmoid_focal_loss\nfrom torchvision.models.detection.retinanet import RetinaNet, RetinaNetClassificationHead\nfrom torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights\nfrom torchvision.models.detection.anchor_utils import AnchorGenerator\n\ndef max_ignore_ioa(boxes, ignore_boxes, chunk_size=4096):\n    boxes = boxes.float()\n    ignore_boxes = ignore_boxes.to(\n        device=boxes.device,\n        dtype=torch.float32,\n    )\n\n    result = boxes.new_zeros(len(boxes))\n\n    if len(boxes) == 0 or len(ignore_boxes) == 0:\n        return result\n\n    for start in range(0, len(boxes), chunk_size):\n        chunk = boxes[start:start + chunk_size]\n\n        top_left = torch.maximum(\n            chunk[:, None, :2],\n            ignore_boxes[None, :, :2],\n        )\n        bottom_right = torch.minimum(\n            chunk[:, None, 2:],\n            ignore_boxes[None, :, 2:],\n        )\n\n        intersection = (\n            (bottom_right - top_left)\n            .clamp_min(0)\n            .prod(dim=-1)\n        )\n\n        anchor_area = (\n            (chunk[:, 2:] - chunk[:, :2])\n            .clamp_min(0)\n            .prod(dim=-1)\n            .clamp_min(1e-6)\n        )\n\n        result[start:start + len(chunk)] = (\n            intersection / anchor_area[:, None]\n        ).amax(dim=1)\n\n    return result\n\ndef exclude_ignored_anchors(\n    matched_indices,\n    anchors,\n    ignore_boxes,\n    threshold=0.5,\n):\n    updated = matched_indices.clone()\n\n    background = (\n        matched_indices == detection_utils.Matcher.BELOW_LOW_THRESHOLD\n    )\n\n    background_indices = torch.where(background)[0]\n\n    if background_indices.numel() > 0:\n        overlaps = max_ignore_ioa(\n            anchors[background_indices],\n            ignore_boxes,\n        )\n\n        excluded_indices = background_indices[overlaps >= threshold]\n\n        updated[excluded_indices] = (\n            detection_utils.Matcher.BETWEEN_THRESHOLDS\n        )\n\n    return updated\n\nclass IgnoreAwareTransform(GeneralizedRCNNTransform):\n    def resize(self, image, target=None):\n        original_size = image.shape[-2:]\n\n        if target is not None:\n            target = target.copy()\n            ignore_boxes = target.get("ignore_boxes")\n        else:\n            ignore_boxes = None\n\n        image, target = super().resize(image, target)\n\n        if ignore_boxes is not None:\n            target["ignore_boxes"] = resize_boxes(\n                ignore_boxes,\n                original_size,\n                image.shape[-2:],\n            )\n\n        return image, target\n\nclass ConfigurableClassificationHead(RetinaNetClassificationHead):\n    focal_alpha = 0.25\n    focal_gamma = 2.0\n\n    def compute_loss(self, targets, head_outputs, matched_idxs):\n        image_losses = []\n\n        for target, logits, matches in zip(\n            targets,\n            head_outputs["cls_logits"],\n            matched_idxs,\n        ):\n            logits = logits.float()\n\n            positive = matches >= 0\n            valid = matches != self.BETWEEN_THRESHOLDS\n\n            class_targets = torch.zeros_like(logits)\n\n            class_targets[\n                positive,\n                target["labels"][matches[positive]],\n            ] = 1.0\n\n            loss = sigmoid_focal_loss(\n                logits[valid],\n                class_targets[valid],\n                alpha=self.focal_alpha,\n                gamma=self.focal_gamma,\n                reduction="sum",\n            )\n\n            normalizer = positive.sum().clamp_min(1).to(logits.dtype)\n            image_losses.append(loss / normalizer)\n\n        return torch.stack(image_losses).mean()\n\nclass IgnoreAwareRetinaNet(RetinaNet):\n    ignore_ioa_threshold = 0.5\n\n    def compute_loss(self, targets, head_outputs, anchors):\n        matched_indices = []\n\n        for target, anchors_per_image in zip(targets, anchors):\n            if target["boxes"].numel() == 0:\n                matches = torch.full(\n                    (len(anchors_per_image),),\n                    detection_utils.Matcher.BELOW_LOW_THRESHOLD,\n                    dtype=torch.int64,\n                    device=anchors_per_image.device,\n                )\n            else:\n                quality = box_iou(\n                    target["boxes"].float(),\n                    anchors_per_image.float(),\n                )\n                matches = self.proposal_matcher(quality)\n\n            matches = exclude_ignored_anchors(\n                matches,\n                anchors_per_image,\n                target["ignore_boxes"],\n                threshold=self.ignore_ioa_threshold,\n            )\n            matched_indices.append(matches)\n\n        return self.head.compute_loss(\n            targets,\n            head_outputs,\n            anchors,\n            matched_indices,\n        )\n\ndef build_retinanet(config, pretrained=True):\n    if config["num_classes"] != 8:\n        raise ValueError("Expected eight target classes.")\n\n    model = retinanet_resnet50_fpn_v2(\n        weights=(\n            RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT\n            if pretrained else None\n        ),\n        weights_backbone=None,\n        min_size=config["min_size"],\n        max_size=config["max_size"],\n        score_thresh=config["score_thresh"],\n        nms_thresh=config["nms_thresh"],\n        detections_per_img=config["detections_per_img"],\n        topk_candidates=config["topk_candidates"],\n    )\n\n    for name, parameter in model.backbone.body.named_parameters():\n        parameter.requires_grad_(\n            name.startswith(("layer2.", "layer3.", "layer4."))\n        )\n\n    anchor_sizes = tuple(\n        (\n            size,\n            int(size * 2 ** (1 / 3)),\n            int(size * 2 ** (2 / 3)),\n        )\n        for size in config["anchor_base_sizes"]\n    )\n\n    anchor_generator = AnchorGenerator(\n        sizes=anchor_sizes,\n        aspect_ratios=(\n            tuple(config["aspect_ratios"]),\n        ) * len(anchor_sizes),\n    )\n\n    anchors_per_level = anchor_generator.num_anchors_per_location()\n    original_count = model.head.classification_head.num_anchors\n\n    if len(anchor_sizes) != 5:\n        raise ValueError("The model expects five feature levels.")\n\n    if any(count != original_count for count in anchors_per_level):\n        raise ValueError(\n            "Preserve the pretrained head\'s anchor count per location."\n        )\n\n    model.anchor_generator = anchor_generator\n\n    classification_head = model.head.classification_head\n    old_output = classification_head.cls_logits\n\n    new_output = nn.Conv2d(\n        in_channels=old_output.in_channels,\n        out_channels=original_count * config["num_classes"],\n        kernel_size=3,\n        padding=1,\n    )\n\n    nn.init.normal_(new_output.weight, std=0.01)\n    nn.init.constant_(\n        new_output.bias,\n        -math.log((1 - 0.01) / 0.01),\n    )\n\n    classification_head.cls_logits = new_output\n    classification_head.num_classes = config["num_classes"]\n    classification_head.__class__ = ConfigurableClassificationHead\n    classification_head.focal_alpha = config["focal_alpha"]\n    classification_head.focal_gamma = config["focal_gamma"]\n\n    model.transform.__class__ = IgnoreAwareTransform\n    model.__class__ = IgnoreAwareRetinaNet\n    model.ignore_ioa_threshold = config["ignore_ioa_threshold"]\n\n    assert model.head.regression_head._loss_type == "giou"\n\n    return model'
exec(RETINA_SOURCE, retina_ns)
print('RetinaNet builder ready')


RetinaNet builder ready


## 3. Dataset and checkpoint integrity

We use the **same** RGB `[0,1]` tensors for both detectors. The model's own transform applies the resizing and normalization stored in the checkpoint configuration. The output label conventions differ internally (Faster R-CNN: 1–8 with background 0; RetinaNet: 0–7), so RetinaNet labels are shifted **+1 only when converting predictions to COCO category IDs**.

In [20]:
class OfficialValDataset(Dataset):
    def __init__(self, table, root):
        self.table = table.reset_index(drop=True)
        self.root = Path(root)
    def __len__(self): return len(self.table)
    def __getitem__(self, index):
        row = self.table.iloc[index]
        with Image.open(self.root / row.file_name) as image:
            image = image.convert('RGB')
            assert image.size == (int(row.width), int(row.height))
            tensor = pil_to_tensor(image).float().div_(255)
        return tensor, int(row.image_id)

val_dataset = OfficialValDataset(val_table, DATA_ROOT/'val')

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fh:
        for block in iter(lambda: fh.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def load_model(name, path):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)  # trusted own checkpoints only
    assert 'identity' in ckpt and 'model' in ckpt, f'Unexpected checkpoint schema: {path}'
    config = ckpt['identity']['model']
    if name.startswith('Faster'):
        assert config['num_classes'] == 9
        model = faster_ns['build_faster_rcnn'](config, pretrained=False)
    else:
        assert config['num_classes'] == 8
        model = retina_ns['build_retinanet'](config, pretrained=False)
    model.load_state_dict(ckpt['model'], strict=True)
    del ckpt
    return model.to(DEVICE).eval(), config

for name, path in CHECKPOINTS.items():
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    print(name, '| config:', ckpt['identity']['model'])
    del ckpt
    gc.collect()


Faster R-CNN baseline | config: {'num_classes': 9, 'min_size': 640, 'max_size': 1152, 'box_detections_per_img': 500, 'box_score_thresh': 0.001, 'ignore_ioa_threshold': 0.5, 'rpn_post_nms_top_n_train': 2000, 'rpn_post_nms_top_n_test': 1000, 'box_batch_size_per_image': 512, 'box_positive_fraction': 0.25, 'roi_box_loss': 'smooth_l1'}
Faster R-CNN tuned | config: {'num_classes': 9, 'min_size': 640, 'max_size': 1152, 'box_detections_per_img': 500, 'box_score_thresh': 0.001, 'ignore_ioa_threshold': 0.5, 'rpn_post_nms_top_n_train': 4000, 'rpn_post_nms_top_n_test': 2000, 'box_batch_size_per_image': 512, 'box_positive_fraction': 0.25, 'roi_box_loss': 'giou'}
RetinaNet baseline | config: {'model': 'retinanet_resnet50_fpn_v2', 'num_classes': 8, 'min_size': 640, 'max_size': 1152, 'anchor_base_sizes': [32, 64, 128, 256, 512], 'aspect_ratios': [0.5, 1.0, 2.0], 'focal_alpha': 0.25, 'focal_gamma': 2.0, 'ignore_ioa_threshold': 0.5, 'score_thresh': 0.001, 'nms_thresh': 0.5, 'detections_per_img': 500, 't

## 4. Original ignore-aware COCO evaluator

This reproduces the **per-class ignored-region duplication logic** of the original RetinaNet evaluator, rather than naively dropping `iscrowd` boxes. It computes mAP, AP50, AP_small, AP_medium and AP_large with the same `max_dets=100`. This approach is memory-conscious because it evaluates one category at a time.

In [21]:
EVAL_SOURCE = 'class LazyGroundTruth:\n    def __init__(self, source, image_ids):\n        self.source = source\n        self.image_ids = set(map(int, image_ids))\n        assert self.image_ids.issubset({int(x[\'id\']) for x in source[\'images\']})\n        self.categories = sorted(source[\'categories\'], key=lambda x: x[\'id\'])\n\ndef build_eval_coco(source, image_ids):\n    # Retain references, rather than duplicating all ignored regions eight times.\n    return LazyGroundTruth(source, image_ids)\n\ndef _atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    with temporary.open(\'w\') as file:\n        json.dump(value, file, sort_keys=True, allow_nan=False)\n        file.flush()\n        os.fsync(file.fileno())\n    os.replace(temporary, path)\n\nclass PredictionCache:\n    # Each row: image_id, category_id, score, x, y, width, height.\n    def __init__(self, directory):\n        self.directory = Path(directory)\n\n    def chunks(self):\n        return sorted(self.directory.glob(\'part_*.npz\'))\n\n    def rows_for_category(self, category_id):\n        for path in self.chunks():\n            with np.load(path, allow_pickle=False) as part:\n                rows = part[\'rows\']\n                yield rows[rows[:, 1] == category_id]\n\ndef _category_gt(ground_truth, category_id):\n    source = ground_truth.source\n    annotations = []\n    for original in source[\'annotations\']:\n        if int(original[\'image_id\']) not in ground_truth.image_ids:\n            continue\n        ignored = bool(original.get(\'iscrowd\', 0) or original.get(\'ignore\', 0))\n        if not ignored and original[\'category_id\'] != category_id:\n            continue\n        x, y, w, h = original[\'bbox\']\n        annotations.append({\n            \'id\': len(annotations) + 1,\n            \'image_id\': int(original[\'image_id\']),\n            \'category_id\': category_id,\n            \'bbox\': [x, y, w, h],\n            \'area\': w * h,\n            \'iscrowd\': int(ignored),\n        })\n    coco = COCO()\n    coco.dataset = {\n        \'info\': {},\n        \'images\': [x for x in source[\'images\'] if int(x[\'id\']) in ground_truth.image_ids],\n        \'categories\': [x for x in ground_truth.categories if x[\'id\'] == category_id],\n        \'annotations\': annotations,\n    }\n    with contextlib.redirect_stdout(io.StringIO()):\n        coco.createIndex()\n    return coco\n\ndef _prediction_rows(predictions, category_id):\n    if isinstance(predictions, PredictionCache):\n        yield from predictions.rows_for_category(category_id)\n    else:\n        rows = [[p[\'image_id\'], p[\'category_id\'], p[\'score\'], *p[\'bbox\']]\n                for p in predictions if p[\'category_id\'] == category_id]\n        yield np.asarray(rows, dtype=np.float64).reshape(-1, 7)\n\ndef _top_per_image(rows, max_dets):\n    if not len(rows):\n        return rows\n    # Stable score order matches COCO\'s mergesort, including tied scores.\n    order = np.lexsort((np.arange(len(rows)), -rows[:, 2], rows[:, 0]))\n    ordered_ids = rows[order, 0]\n    starts = np.flatnonzero(np.r_[True, ordered_ids[1:] != ordered_ids[:-1]])\n    ranks = np.arange(len(order)) - np.repeat(starts, np.diff(np.r_[starts, len(order)]))\n    keep = order[ranks < max_dets]\n    return rows[np.sort(keep)]\n\ndef evaluate_coco_predictions(coco_gt, predictions, max_dets=100):\n    if max_dets < 10:\n        raise ValueError(\'max_dets must be at least 10.\')\n    if not isinstance(coco_gt, LazyGroundTruth):\n        raise TypeError(\'Re-run build_eval_coco after replacing this cell.\')\n    started = time.perf_counter()\n    subset_key = hashlib.sha256(json.dumps(sorted(coco_gt.image_ids)).encode()).hexdigest()[:16]\n    selected_image_ids = np.array(sorted(coco_gt.image_ids), dtype=np.int64)\n    metric_names = [\'mAP\', \'AP50\', \'AP75\', \'AP_small\', \'AP_medium\', \'AP_large\']\n    totals = {name: [0.0, 0] for name in metric_names}\n    per_class = {}\n    for category in coco_gt.categories:\n        category_id = category[\'id\']\n        result_path = None\n        result = None\n        if isinstance(predictions, PredictionCache):\n            result_path = predictions.directory / f\'metrics_v2_subset{subset_key}_cat{category_id}_max{max_dets}.json\'\n            if result_path.is_file():\n                result = json.loads(result_path.read_text())\n        if result is None:\n            gt = _category_gt(coco_gt, category_id)\n            parts = [_top_per_image(rows[np.isin(rows[:, 0], selected_image_ids)], max_dets)\n                     for rows in _prediction_rows(predictions, category_id)]\n            rows = np.concatenate(parts) if parts else np.empty((0, 7))\n            del parts\n            # Cache chunks never split an image; this second cap also supports plain lists.\n            rows = _top_per_image(rows, max_dets)\n            detections = [dict(image_id=int(r[0]), category_id=int(r[1]),\n                               score=float(r[2]), bbox=r[3:].tolist()) for r in rows]\n            del rows\n            print(f"COCO class {category[\'name\']}: {len(detections):,} predictions", flush=True)\n            with contextlib.redirect_stdout(io.StringIO()):\n                if detections:\n                    dt = gt.loadRes(detections)\n                else:\n                    dt = COCO()\n                    dt.dataset = {**gt.dataset, \'annotations\': []}\n                    dt.createIndex()\n                evaluator = COCOeval(gt, dt, \'bbox\')\n                evaluator.params.imgIds = sorted(coco_gt.image_ids)\n                evaluator.params.catIds = [category_id]\n                evaluator.params.maxDets = [1, 10, max_dets]\n                evaluator.evaluate()\n                evaluator.accumulate()\n            precision = evaluator.eval[\'precision\']\n            area = evaluator.params.areaRngLbl\n            selections = {\n                \'mAP\': precision[:, :, :, area.index(\'all\'), -1],\n                \'AP50\': precision[np.isclose(evaluator.params.iouThrs, .5), :, :, area.index(\'all\'), -1],\n                \'AP75\': precision[np.isclose(evaluator.params.iouThrs, .75), :, :, area.index(\'all\'), -1],\n                **{f\'AP_{a}\': precision[:, :, :, area.index(a), -1] for a in [\'small\', \'medium\', \'large\']},\n            }\n            result = {}\n            for name, values in selections.items():\n                valid = values[values >= 0]\n                result[name] = [float(valid.sum()), int(valid.size)]\n            del gt, dt, evaluator, precision, selections, detections\n            gc.collect()\n            if result_path is not None:\n                _atomic_json(result_path, result)\n        else:\n            print(f"COCO class {category[\'name\']}: using saved metrics", flush=True)\n        for name in metric_names:\n            totals[name][0] += result[name][0]\n            totals[name][1] += result[name][1]\n        total, count = result[\'mAP\']\n        per_class[category[\'name\']] = total / count if count else float(\'nan\')\n    metrics = {name: total / count if count else float(\'nan\')\n               for name, (total, count) in totals.items()}\n    metrics[\'max_dets\'] = max_dets\n    print(f\'COCO evaluation finished in {(time.perf_counter() - started) / 60:.2f} min.\')\n    return metrics, per_class'
exec(EVAL_SOURCE, globals())
val_gt = build_eval_coco(official_val_coco, val_table.image_id)
print('Ignore-aware evaluator ready; categories:', [c['name'] for c in val_gt.categories])


Ignore-aware evaluator ready; categories: ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']


## 5. Resumable inference for all official validation images

Each completed chunk is saved as `part_XXXXX.npz` containing one row per detection: `[image_id, category_id, score, x, y, width, height]`. On restart, existing chunks are reused **only if the checkpoint hash, dataset manifest, and inference settings match**. Do not change the image set halfway through a run.

In [22]:
def inference_signature(name, path):
    payload = {'name':name, 'checkpoint_sha256':sha256_file(path),
               'images_sha256':hashlib.sha256(val_table.to_csv(index=False).encode()).hexdigest(),
               'torch':torch.__version__, 'torchvision':torchvision.__version__,
               'precision':'fp32', 'max_dets':MAX_DETS, 'chunk_frames':CHUNK_FRAMES}
    return payload

def run_validation_inference(name, path):
    signature = inference_signature(name,path)
    key = hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()[:16]
    directory = OUTPUT_DIR / 'predictions' / key
    directory.mkdir(parents=True,exist_ok=True)
    manifest_path = directory/'manifest.json'
    image_ids = val_table.image_id.astype(int).tolist()
    manifest = {**signature,'image_ids':image_ids,'chunk_frames':CHUNK_FRAMES}
    if manifest_path.exists():
        assert json.loads(manifest_path.read_text()) == manifest, 'Cache manifest mismatch.'
    else:
        manifest_path.write_text(json.dumps(manifest,indent=2))
    expected = math.ceil(len(val_dataset)/CHUNK_FRAMES)
    missing = [i for i in range(expected) if not (directory/f'part_{i:05d}.npz').exists()]
    print(f'{name}: {expected-len(missing)}/{expected} chunks already cached')
    if missing:
        model, config = load_model(name,path)
        try:
            with torch.inference_mode():
                for chunk_id in tqdm(missing,desc=name):
                    start=chunk_id*CHUNK_FRAMES
                    stop=min(len(val_dataset),start+CHUNK_FRAMES)
                    rows=[]; chunk_ids=[]
                    for i in range(start,stop):
                        image, image_id = val_dataset[i]
                        output = model([image.to(DEVICE)])[0]
                        boxes=output['boxes'].detach().cpu().numpy()
                        labels=output['labels'].detach().cpu().numpy().astype(int)
                        scores=output['scores'].detach().cpu().numpy()
                        if name.startswith('RetinaNet'): labels=labels+1
                        assert np.all((labels>=1)&(labels<=8)), (name,np.unique(labels))
                        for b,cat,score in zip(boxes,labels,scores):
                            x1,y1,x2,y2 = map(float,b)
                            rows.append([image_id,cat,float(score),x1,y1,max(0.,x2-x1),max(0.,y2-y1)])
                        chunk_ids.append(image_id)
                    arr=np.asarray(rows,dtype=np.float64).reshape(-1,7)
                    tmp=directory/f'part_{chunk_id:05d}.tmp.npz'
                    np.savez_compressed(tmp, image_ids=np.asarray(chunk_ids,dtype=np.int64),rows=arr)
                    os.replace(tmp,directory/f'part_{chunk_id:05d}.npz')
        finally:
            del model
            gc.collect();torch.cuda.empty_cache()
    # Validate coverage and no stale chunks
    parts=sorted(directory.glob('part_*.npz'))
    assert len(parts)==expected
    for i,p in enumerate(parts):
        with np.load(p,allow_pickle=False) as chunk:
            ids=chunk['image_ids'].tolist()
            assert ids==image_ids[i*CHUNK_FRAMES:(i+1)*CHUNK_FRAMES]
            assert chunk['rows'].shape[1]==7 and np.isfinite(chunk['rows']).all()
    return PredictionCache(directory)

prediction_caches = {}
for name, path in CHECKPOINTS.items():
    prediction_caches[name]=run_validation_inference(name,path)
print('All official validation predictions complete.')


Faster R-CNN baseline: 89/89 chunks already cached
Faster R-CNN tuned: 89/89 chunks already cached
RetinaNet baseline: 89/89 chunks already cached
RetinaNet smaller anchors: 34/89 chunks already cached


RetinaNet smaller anchors:   0%|          | 0/55 [00:00<?, ?it/s]

All official validation predictions complete.


## 6. Official validation COCO metrics

Metrics are calculated on **all 2,846 official validation frames**, with the same GT, evaluator and `max_dets=100`. This may take several minutes per model. We save metrics after each candidate so a restart does not discard completed results.

In [23]:
metric_path=OUTPUT_DIR/'official_validation_metrics.csv'
records = pd.read_csv(metric_path).to_dict('records') if metric_path.exists() else []
completed={r['candidate'] for r in records}
for name, cache in prediction_caches.items():
    if name in completed:
        print('Reusing official validation metrics:',name);continue
    metrics,per_class=evaluate_coco_predictions(val_gt,cache,max_dets=MAX_DETS)
    record={'candidate':name,'split':'official_val','fold':0,**metrics}
    records.append(record)
    pd.DataFrame(records).to_csv(metric_path,index=False)
    (OUTPUT_DIR/f'per_class_{name.lower().replace(" ","_")}.json').write_text(json.dumps(per_class,indent=2))
metrics_table=pd.DataFrame(records)
display(metrics_table[['candidate','mAP','AP50','AP_small','AP_medium','AP_large','max_dets']])


COCO class person: 194,031 predictions
COCO class bicycle: 60,547 predictions
COCO class car: 158,991 predictions
COCO class van: 39,165 predictions
COCO class truck: 44,447 predictions
COCO class tricycle: 45,081 predictions
COCO class bus: 23,325 predictions
COCO class motor: 100,926 predictions
COCO evaluation finished in 5.00 min.
COCO class person: 142,879 predictions
COCO class bicycle: 40,569 predictions
COCO class car: 82,413 predictions
COCO class van: 17,716 predictions
COCO class truck: 17,500 predictions
COCO class tricycle: 29,595 predictions
COCO class bus: 7,992 predictions
COCO class motor: 65,192 predictions
COCO evaluation finished in 3.68 min.
COCO class person: 259,923 predictions
COCO class bicycle: 111,635 predictions
COCO class car: 243,469 predictions
COCO class van: 122,461 predictions
COCO class truck: 139,697 predictions
COCO class tricycle: 106,123 predictions
COCO class bus: 119,168 predictions
COCO class motor: 168,542 predictions
COCO evaluation finished 

,candidate,mAP,AP50,AP_small,AP_medium,AP_large,max_dets
0,Faster R-CNN baseline,0.168492,0.343421,0.065114,0.164735,0.319328,100
1,Faster R-CNN tuned,0.173837,0.350219,0.074241,0.168660,0.319824,100
2,RetinaNet baseline,0.151721,0.302021,0.052879,0.158991,0.265586,100
3,RetinaNet smaller anchors,0.151295,0.334956,0.069745,0.141452,0.254822,100


## 7. Same-GPU model-only inference speed

Each model is loaded sequentially to avoid GPU memory interference. Same GPU, same 64 evenly spaced validation images, same batch size 1, FP32, five warmup passes. **Model-only latency includes GPU transfer, model forward/postprocessing and output transfer to CPU**, but excludes disk image loading. End-to-end latency includes image loading. Both are reported; avoid comparing these numbers with previously measured FPS on other GPUs.

In [24]:
@torch.inference_mode()
def measure_speed(name,path,indices):
    model,config=load_model(name,path)
    latencies=[];end_to_end=[]
    try:
        image,_=val_dataset[int(indices[0])]
        for _ in range(WARMUP):
            result=model([image.to(DEVICE)])
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        for idx in tqdm(indices,desc='FPS '+name,leave=False):
            full_start=time.perf_counter()
            image,_=val_dataset[int(idx)]
            torch.cuda.synchronize()
            start=time.perf_counter()
            result=model([image.to(DEVICE)])
            _=[{k:v.cpu() for k,v in item.items()} for item in result]
            torch.cuda.synchronize()
            latencies.append(time.perf_counter()-start)
            end_to_end.append(time.perf_counter()-full_start)
        return {'candidate':name,'fps':1/float(np.mean(latencies)),
                'mean_ms':1000*float(np.mean(latencies)),
                'p50_ms':1000*float(np.median(latencies)),
                'p95_ms':1000*float(np.quantile(latencies,.95)),
                'end_to_end_fps':1/float(np.mean(end_to_end)),
                'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,
                'gpu':torch.cuda.get_device_name(DEVICE),
                'torch':torch.__version__,'torchvision':torchvision.__version__,
                'batch_size':1,'precision':'fp32','images':len(indices),
                'min_size':config['min_size'],'max_size':config['max_size']}
    finally:
        del model;gc.collect();torch.cuda.empty_cache()

speed_path=OUTPUT_DIR/'speed.csv'
speed_records=pd.read_csv(speed_path).to_dict('records') if speed_path.exists() else []
speed_done={x['candidate'] for x in speed_records}
indices=np.linspace(0,len(val_dataset)-1,min(BENCHMARK_IMAGES,len(val_dataset)),dtype=int)
for name,path in CHECKPOINTS.items():
    if name in speed_done:continue
    result=measure_speed(name,path,indices)
    speed_records.append(result)
    pd.DataFrame(speed_records).to_csv(speed_path,index=False)
speed_table=pd.DataFrame(speed_records)
display(speed_table[['candidate','fps','mean_ms','p95_ms','end_to_end_fps','gpu']])


FPS Faster R-CNN baseline:   0%|          | 0/64 [00:00<?, ?it/s]

FPS Faster R-CNN tuned:   0%|          | 0/64 [00:00<?, ?it/s]

FPS RetinaNet baseline:   0%|          | 0/64 [00:00<?, ?it/s]

FPS RetinaNet smaller anchors:   0%|          | 0/64 [00:00<?, ?it/s]

,candidate,fps,mean_ms,p95_ms,end_to_end_fps,gpu
0,Faster R-CNN baseline,7.635590,130.965646,143.637665,5.022575,Tesla T4
1,Faster R-CNN tuned,5.996359,166.767881,185.808427,4.325455,Tesla T4
2,RetinaNet baseline,11.002110,90.891659,102.327704,6.424921,Tesla T4
3,RetinaNet smaller anchors,10.924172,91.540115,104.914931,6.319766,Tesla T4


## 8. Final comparison and interpretation

- Faster R-CNN: Stage 1 (RPN) proposes class-agnostic candidate regions; Stage 2 (RoI head) classifies and refines them.
- RetinaNet: no separate proposal stage; dense anchor classification introduces foreground/background imbalance, mitigated with focal loss. Smaller anchors may help small objects but can hurt total mAP.
- **Do not claim tuned RetinaNet wins:** its baseline was selected by fold-0 mAP; the smaller-anchor variant is an informative tuning candidate.
- Failure-case visualizations and explanations remain a separate deliverable in the Faster R-CNN notebook/report.
- Fold-0 checkpoint evaluation is **not** the same as reporting the 3-fold mean; preserve both results separately.

In [25]:
final=metrics_table.merge(speed_table,on='candidate',validate='one_to_one')
final=final[['candidate','split','fold','mAP','AP50','AP_small','AP_medium','AP_large','fps','mean_ms','p95_ms','end_to_end_fps','gpu']]
final.to_csv(OUTPUT_DIR/'final_anchor_based_comparison.csv',index=False)
display(final.round(4))

fr=final.set_index('candidate')
for other,base in [('Faster R-CNN tuned','Faster R-CNN baseline'),
                   ('RetinaNet smaller anchors','RetinaNet baseline'),
                   ('RetinaNet baseline','Faster R-CNN tuned')]:
    print(f'{other} − {base}: ΔmAP={fr.loc[other,"mAP"]-fr.loc[base,"mAP"]:+.4f}, '
          f'ΔAP_small={fr.loc[other,"AP_small"]-fr.loc[base,"AP_small"]:+.4f}')
print('Saved:',OUTPUT_DIR/'final_anchor_based_comparison.csv')


,candidate,split,fold,mAP,AP50,AP_small,AP_medium,AP_large,fps,mean_ms,p95_ms,end_to_end_fps,gpu
0,Faster R-CNN baseline,official_val,0,0.1685,0.3434,0.0651,0.1647,0.3193,7.6356,130.9656,143.6377,5.0226,Tesla T4
1,Faster R-CNN tuned,official_val,0,0.1738,0.3502,0.0742,0.1687,0.3198,5.9964,166.7679,185.8084,4.3255,Tesla T4
2,RetinaNet baseline,official_val,0,0.1517,0.3020,0.0529,0.1590,0.2656,11.0021,90.8917,102.3277,6.4249,Tesla T4
3,RetinaNet smaller anchors,official_val,0,0.1513,0.3350,0.0697,0.1415,0.2548,10.9242,91.5401,104.9149,6.3198,Tesla T4


Faster R-CNN tuned − Faster R-CNN baseline: ΔmAP=+0.0053, ΔAP_small=+0.0091
RetinaNet smaller anchors − RetinaNet baseline: ΔmAP=-0.0004, ΔAP_small=+0.0169
RetinaNet baseline − Faster R-CNN tuned: ΔmAP=-0.0221, ΔAP_small=-0.0214
Saved: /content/drive/MyDrive/ucu_ai_hw2/benchmarks/anchor_based_comparison/final_anchor_based_comparison.csv
